<a href="https://colab.research.google.com/github/Future-Analyst/Tensorflow-Exercises-/blob/main/04_feature_extrator_exr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Transfer Learning Exr[festure extractor]

Transfer learning is a machine learning technique where a model trained on one task is reused for a different but related task, allowing it to apply previously learned features and patterns instead of starting from scratch. This approach saves time, reduces the amount of data needed, and improves performance, especially in problems like image classification using models such as MobileNetV2 or ResNet.

The two main types are feature extraction and fine-tuning. In feature extraction, most of the pre-trained model is kept frozen and only the final layers are trained, making it faster and suitable for small datasets. In contrast, fine-tuning involves unfreezing some or all layers and retraining them, which makes the model more adaptable and accurate but requires more data and time; therefore, feature extraction is simpler and quicker, while fine-tuning is more flexible but computationally intensive.



In [1]:
# add timestamp
import datetime
print(f"Notebook last run (end-to-end): {datetime.datetime.now()}")

Notebook last run (end-to-end): 2026-04-05 01:45:11.555830


In [3]:
# Check the GPU
!nvidia-smi

/bin/bash: line 1: nvidia-smi: command not found


In [5]:
# Importing the necessary dependencies
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

import tensorflow as tf
from tensorflow.keras import layers
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Dense, Flatten, MaxPooling2D
from tensorflow.keras.losses import CategoricalCrossentropy
from tensorflow.keras.optimizers import SGD ,Adam
from tensorflow.keras.layers import BatchNormalization, Dropout


## Downloading and becoming one with the data

In [9]:
# Configure kaggle
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [16]:
import kagglehub

# Download the dataset
path = kagglehub.dataset_download("tongpython/cat-and-dog")

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'cat-and-dog' dataset.
Path to dataset files: /kaggle/input/cat-and-dog


In [18]:
# The dataset has already been downloaded and unzipped into the directory:
# /kaggle/input/cat-and-dog


# You can verify the contents of the directory:
import os
print(os.listdir('/kaggle/input/cat-and-dog'))


['test_set', 'training_set']


### Get 10% of the Data

In [22]:
import os
import random
from glob import glob

# Path to dataset (from your previous download)
dataset_path = path

# Let's inspect the contents of the training_set to confirm the directory structure
print(f"Contents of {os.path.join(dataset_path, 'training_set')}: {os.listdir(os.path.join(dataset_path, 'training_set'))}")

# Get all cat and dog images from training_set and test_set
# Correcting the path for the nested directory structure
cat_images = glob(os.path.join(dataset_path, "training_set", "training_set", "cats", "*")) + \
             glob(os.path.join(dataset_path, "test_set", "test_set", "cats", "*"))
dog_images = glob(os.path.join(dataset_path, "training_set", "training_set", "dogs", "*")) + \
             glob(os.path.join(dataset_path, "test_set", "test_set", "dogs", "*"))

# Set seed for reproducibility
random.seed(42)

# Take 10% of each (ensure at least 1 image is sampled if available)
cat_sample = random.sample(cat_images, max(1, len(cat_images)//10)) if cat_images else []
dog_sample = random.sample(dog_images, max(1, len(dog_images)//10)) if dog_images else []

print(f"Number of cat images selected: {len(cat_sample)}")
print(f"Number of dog images selected: {len(dog_sample)}")

# Combine into one dataset if needed
sample_images = cat_sample + dog_sample


Contents of /kaggle/input/cat-and-dog/training_set: ['training_set']
Number of cat images selected: 501
Number of dog images selected: 501


In [23]:

# Path to the full dataset
dataset_path = path
# Destination folder for the 10% subset
subset_path = "./cat_dog_10_percent"
os.makedirs(subset_path, exist_ok=True)

# Create train/test subfolders
for split in ["train", "test"]:
    for cls in ["cats", "dogs"]:
        os.makedirs(os.path.join(subset_path, split, cls), exist_ok=True)

In [26]:
# How many images in each folder?
import os

# Walk through 10 percent data directory and list number of files
for dirpath, dirnames, filenames in os.walk("/content/cat_dog_10_percent"):
  print(f"There are {len(dirnames)} directories and {len(filenames)} images in '{dirpath}'.")

There are 2 directories and 0 images in '/content/cat_dog_10_percent'.
There are 2 directories and 0 images in '/content/cat_dog_10_percent/train'.
There are 0 directories and 0 images in '/content/cat_dog_10_percent/train/cats'.
There are 0 directories and 0 images in '/content/cat_dog_10_percent/train/dogs'.
There are 2 directories and 0 images in '/content/cat_dog_10_percent/test'.
There are 0 directories and 0 images in '/content/cat_dog_10_percent/test/cats'.
There are 0 directories and 0 images in '/content/cat_dog_10_percent/test/dogs'.


## Create Data Loaders (preparing the data)

now we have downloaded the data successfuly , lets use the imageDataGenerator clas along with the flow_from_directory method to load in our images


In [30]:
# setup the data inputs

from tensorflow.keras.preprocessing.image import ImageDataGenerator

IMAGE_SHAPE = (224, 224)
BATCH_SIZE = 32

train_dir = "./cat_dog_10_percent/train/"
test_dir = "./cat_dog_10_percent/test/"

train_datagen = ImageDataGenerator(rescale=1/255.)
test_datagen =ImageDataGenerator(rescale=1/255.)

print("Training images:")
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SHAPE,
    batch_size = BATCH_SIZE,
    class_mode = "categorical"
)

print("Testing images:")
train_data = train_datagen.flow_from_directory(
    train_dir,
    target_size=IMAGE_SHAPE,
    batch_size = BATCH_SIZE,
    class_mode = "categorical"
)

Training images:
Found 0 images belonging to 2 classes.
Testing images:
Found 0 images belonging to 2 classes.
